# Anexo · Voz con Whisper: graba, transcribe, traduce y subtitula
*Curso «Google Colab» · Anexo multimodal · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

**Objetivo:** grabar tu voz con el micrófono del navegador desde Colab, convertirla en texto con **Whisper**,
traducirla al inglés y generar subtítulos. Todo con modelos abiertos, en la GPU gratis, **sin claves**.

| Modelo | Para qué | Licencia |
|---|---|---|
| `openai/whisper-large-v3-turbo` | Transcribir (con GPU) | MIT |
| `openai/whisper-medium` | Traducir al inglés (con GPU) | Apache 2.0 |
| `openai/whisper-small` / `whisper-tiny` | Transcribir y traducir, más liviano (sin GPU) | Apache 2.0 |
| `facebook/mms-tts-spa` | Texto a voz en español (opcional) | CC BY-NC 4.0: solo uso no comercial |

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os, time
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"   # solo para la prueba automática de MiAtlas.cl

import torch
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
TIPO = torch.float16 if DISPOSITIVO == "cuda" else torch.float32
print("Colab:", EN_COLAB, "| dispositivo:", DISPOSITIVO)
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.")

In [ ]:
%pip install -q transformers accelerate soundfile

## 1. Tu voz, o un audio de ejemplo

La celda siguiente pide permiso al navegador para usar el micrófono y graba unos segundos. Si no tienes micrófono (o
no estás en Colab), se usa una grabación de ejemplo: una consulta a un banco en español de España, del conjunto
**MINDS-14** de PolyAI (CC BY 4.0).

In [ ]:
import io, base64, subprocess
import numpy as np
import soundfile as sf

FRECUENCIA = 16000      # Whisper trabaja con audio a 16 kHz

def grabar(segundos=6):
    """Graba con el micrófono del navegador (solo en Colab) y devuelve el audio a 16 kHz."""
    from google.colab.output import eval_js
    from IPython.display import display, Javascript
    display(Javascript("""
      async function grabar(ms) {
        const flujo = await navigator.mediaDevices.getUserMedia({audio: true});
        const grabadora = new MediaRecorder(flujo); const trozos = [];
        grabadora.ondataavailable = e => trozos.push(e.data);
        grabadora.start(); await new Promise(r => setTimeout(r, ms)); grabadora.stop();
        await new Promise(r => grabadora.onstop = r); flujo.getTracks().forEach(t => t.stop());
        const datos = await new Blob(trozos).arrayBuffer();
        let binario = ""; new Uint8Array(datos).forEach(b => binario += String.fromCharCode(b));
        return btoa(binario);
      }"""))
    print(f"🎙 Grabando {segundos} s… habla ahora.")
    webm = base64.b64decode(eval_js(f"grabar({segundos * 1000})"))
    wav = subprocess.run(["ffmpeg", "-loglevel", "error", "-i", "pipe:0", "-ac", "1", "-ar", str(FRECUENCIA), "-f", "wav", "pipe:1"],
                         input=webm, capture_output=True, check=True).stdout
    audio, _ = sf.read(io.BytesIO(wav), dtype="float32")
    return audio

def audio_de_ejemplo():
    """Una grabación de MINDS-14 (PolyAI, CC BY 4.0), en español de España."""
    import pandas as pd
    from scipy.signal import resample_poly
    url = "https://huggingface.co/api/datasets/PolyAI/minds14/parquet/es-ES/train/0.parquet"
    fila = pd.read_parquet(url).iloc[3]
    audio, frecuencia = sf.read(io.BytesIO(fila["audio"]["bytes"]), dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    print("Texto real de la grabación:", fila["transcription"])
    return resample_poly(audio, FRECUENCIA, frecuencia).astype("float32")

USAR_MICROFONO = EN_COLAB and not PRUEBA_RAPIDA     # cámbialo a False para usar el audio de ejemplo
audio = None
if USAR_MICROFONO:
    try:
        audio = grabar(6)
    except Exception as e:
        print("No se pudo grabar:", type(e).__name__, "→ se usa el audio de ejemplo")
if audio is None:
    audio = audio_de_ejemplo()
print(f"Audio: {len(audio) / FRECUENCIA:.1f} s")

from IPython.display import Audio, display
display(Audio(audio, rate=FRECUENCIA))

## 2. Transcribir con Whisper

Whisper se entrenó con 680.000 horas de audio en muchos idiomas. Con `language="spanish"` le decimos qué idioma
esperar (si no, lo detecta solo).

In [ ]:
from transformers import pipeline

MODELO = "openai/whisper-large-v3-turbo" if DISPOSITIVO == "cuda" else ("openai/whisper-tiny" if PRUEBA_RAPIDA else "openai/whisper-small")
whisper = pipeline("automatic-speech-recognition", model=MODELO, torch_dtype=TIPO, device=DISPOSITIVO)
print("Modelo:", MODELO)

inicio = time.perf_counter()
resultado = whisper({"raw": audio, "sampling_rate": FRECUENCIA}, generate_kwargs={"language": "spanish", "task": "transcribe"})
print(f"Transcripción ({time.perf_counter() - inicio:.1f} s):", resultado["text"])

## 3. Traducir al inglés

Con `task="translate"` Whisper escucha en cualquier idioma y **escribe en inglés** (solo traduce hacia el inglés).

> **Ojo:** el modelo *turbo* **no se entrenó para traducir**: aunque le pidas `translate`, devuelve el texto en el
> idioma original (lo advierte el repositorio oficial de Whisper). Para traducir se usa un modelo multilingüe completo,
> como *medium*.

In [ ]:
TRADUCTOR = "openai/whisper-medium" if DISPOSITIVO == "cuda" else MODELO
traductor = whisper if TRADUCTOR == MODELO else pipeline("automatic-speech-recognition", model=TRADUCTOR, torch_dtype=TIPO, device=DISPOSITIVO)
traduccion = traductor({"raw": audio, "sampling_rate": FRECUENCIA}, generate_kwargs={"language": "spanish", "task": "translate"})
print(f"En inglés ({TRADUCTOR}):", traduccion["text"])

## 4. Subtítulos con marcas de tiempo

`return_timestamps=True` devuelve cada trozo con su inicio y su fin. Con eso armamos un archivo **`.srt`**, el formato
de subtítulos que aceptan los reproductores de video.

In [ ]:
con_tiempos = whisper({"raw": audio, "sampling_rate": FRECUENCIA}, return_timestamps=True,
                      generate_kwargs={"language": "spanish", "task": "transcribe"})

def marca(segundos):
    h, resto = divmod(segundos, 3600); m, s = divmod(resto, 60)
    return f"{int(h):02d}:{int(m):02d}:{s:06.3f}".replace(".", ",")

lineas = []
for n, trozo in enumerate(con_tiempos["chunks"], start=1):
    inicio, fin = trozo["timestamp"]
    fin = fin if fin is not None else len(audio) / FRECUENCIA
    lineas += [str(n), f"{marca(inicio)} --> {marca(fin)}", trozo["text"].strip(), ""]
srt = "\n".join(lineas)
open("subtitulos.srt", "w", encoding="utf-8").write(srt)
print(srt)

## 5. De texto a voz (opcional)

`facebook/mms-tts-spa` lee texto en español. Su licencia es **CC BY-NC 4.0**: sirve para aprender y probar, **no**
para un uso comercial.

In [ ]:
voz = pipeline("text-to-speech", model="facebook/mms-tts-spa", device=DISPOSITIVO)
hablado = voz("Hola, esta voz la generó un modelo abierto en la GPU de Colab.")
display(Audio(hablado["audio"].squeeze(), rate=hablado["sampling_rate"]))

## Pruébalo tú

Transcribe el mismo audio **sin** indicar el idioma (quita `"language"` de `generate_kwargs`) y guarda el texto en
`sin_idioma`. ¿Lo detectó bien?

In [ ]:
sin_idioma = None   # ← tu código

In [ ]:
# Comprobación
if sin_idioma is None:
    print("Aún no lo resuelves.")
elif isinstance(sin_idioma, str) and len(sin_idioma) > 0:
    print("✅ Registrado: compáralo con la transcripción anterior")
else:
    print("❌ Revisa tu respuesta")

## Resumen

- En Colab, el navegador graba y `eval_js` entrega el audio a Python.
- Whisper transcribe (`task="transcribe"`), traduce al inglés (`task="translate"`) y da marcas de tiempo.
- Con GPU, un modelo grande como large-v3-turbo transcribe en segundos (pero no traduce: para eso, *medium* o *large*).
- **Consentimiento:** no grabes ni transcribas a otras personas sin su permiso (lo pide también la ficha del modelo).

**Fuentes:** [Whisper large-v3-turbo (MIT)](https://huggingface.co/openai/whisper-large-v3-turbo) · [Whisper small](https://huggingface.co/openai/whisper-small) · [Whisper medium](https://huggingface.co/openai/whisper-medium) · [Whisper · repositorio oficial (turbo no traduce)](https://github.com/openai/whisper) · [Radford et al. (2022), Whisper](https://arxiv.org/abs/2212.04356) · [Transformers · reconocimiento de voz](https://huggingface.co/docs/transformers/tasks/asr) · [MINDS-14 (CC BY 4.0)](https://huggingface.co/datasets/PolyAI/minds14) · [MMS-TTS español (CC BY-NC 4.0)](https://huggingface.co/facebook/mms-tts-spa) · [MDN · MediaRecorder](https://developer.mozilla.org/es/docs/Web/API/MediaRecorder)